# 🇮🇩 Download & Verify Indonesian KTP Dummy Dataset (Flat Subset)

- **Repository**: [`cloverx-id/indonesian-id-card-dummy`](https://huggingface.co/datasets/cloverx-id/indonesian-id-card-dummy)
- **Subset**: `data/flat/*.parquet` (12 files, ~11.8 GB total, 24,000 cards)
- **Destination**: `datasets/raw/cloverx_ktp/flat/`

This notebook allows downloading the parquet partitions with resume support, followed by an immediate row count and image integrity check.

## 1. Imports & Disk Space Check

In [ ]:
import os
import shutil
from pathlib import Path
import pyarrow.parquet as pq

RAW_DIR = Path("../datasets/raw/cloverx_ktp/flat").resolve()
RAW_DIR.mkdir(parents=True, exist_ok=True)

total, used, free = shutil.disk_usage(RAW_DIR.anchor or ".")
free_gb = free / (1024**3)

print(f"Target directory: {RAW_DIR}")
print(f"Free disk space : {free_gb:.1f} GB (Required for full 12 files: ~11.8 GB)")

## 2. Run Downloader via CLI Script
You can customize `--limit` below:
- `--limit 1`: Downloads only the first partition (~989 MB / 2,000 cards) for a quick test.
- Omit `--limit` or leave empty: Downloads all 12 partitions (~11.8 GB / 24,000 cards).

In [ ]:
# To download all 12 partitions, run:
# !python ../scripts/download_ktp_flat.py --output-dir ../datasets/raw/cloverx_ktp/flat

# Or download partition 0 first as a test:
!python ../scripts/download_ktp_flat.py --output-dir ../datasets/raw/cloverx_ktp/flat --limit 1

## 3. Verify Downloaded Partitions & Count Rows

In [ ]:
parquet_files = sorted(RAW_DIR.glob("*.parquet"))
print(f"Found {len(parquet_files)} parquet file(s) in {RAW_DIR.name}/\n")

total_cards = 0
for pf in parquet_files:
    meta = pq.ParquetFile(pf).metadata
    size_mb = pf.stat().st_size / (1024 * 1024)
    print(f"- {pf.name}: {meta.num_rows:,} cards | {size_mb:.1f} MB")
    total_cards += meta.num_rows

print("-" * 50)
print(f"Total cards downloaded so far: {total_cards:,} / 24,000")

## 4. Inspect Sample Record from Downloaded Parquet

In [ ]:
if parquet_files:
    import pandas as pd
    import matplotlib.pyplot as plt
    import io
    from PIL import Image

    # Read the first 5 rows from the first file
    df = pd.read_parquet(parquet_files[0]).head(5)
    print("Columns:", list(df.columns))
    
    # Display the first image
    first_row = df.iloc[0]
    img_data = first_row["image"]
    
    if isinstance(img_data, dict) and "bytes" in img_data:
        img = Image.open(io.BytesIO(img_data["bytes"]))
    elif isinstance(img_data, (bytes, bytearray)):
        img = Image.open(io.BytesIO(img_data))
    else:
        img = img_data
        
    plt.figure(figsize=(8, 5))
    plt.imshow(img)
    plt.title(f"NIK: {first_row.get('nik')} | {first_row.get('nama')}")
    plt.axis("off")
    plt.show()
else:
    print("No parquet files downloaded yet. Run Section 2 first!")